In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_M05.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.780708992454893, 'n_it': 0.37494439437456906}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 300

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[14.681667695256902, 14.7719299260202, 14.549414761050441, 15.454066380872023, 14.317179957244324, 14.330052993523568, 14.624360868183553, 15.552386027779548, 14.17039126449895, 15.267386111326884, 14.518797146013315, 16.837978189408503, 13.944200345751563, 14.70399671967169, 14.818455630339882, 14.874232513913835, 14.193160104797663, 13.742569199156176, 14.786919489335355, 13.940598158188957, 15.095012673617227, 14.882119538043575, 14.126519926223708, 13.615471784950927, 14.406774130363747, 15.331183305698467, 14.031971002963363, 13.765252754731428, 15.112137122263722, 13.732205950837821, 13.69891282250881, 15.815457186438632, 14.46964585905655, 13.749469239724775, 16.4208984415048, 15.255246051716423, 14.436516791225227, 17.27522341060685, 14.349112821937545, 13.821684765844921, 15.255508243395669, 14.435145919435806, 13.916437001247546, 14.159423413465479, 16.118407142046014, 14.004462350691902, 14.61066811338387, 14.054957763617828, 13.619920465596463, 14.100404520231596, 13.759894

In [5]:
np.average(y_max_arr)

np.float64(14.749318519918955)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M05/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)